# ⚡ Quick Smoke Tests - Fast System Health Check

**Purpose:** Rapid validation of critical SkillUP functionality across all modules

**Target Runtime:** < 30 seconds

**What This Tests:**
- Module imports and basic functionality
- Neo4j connectivity (mocked if credentials unavailable)
- Skill Gap Analyzer smoke test
- Recommender smoke test
- Data file accessibility
- Databricks table availability

**When to Use:**
- Quick health check before committing changes
- After environment setup
- Before running full test suite
- Daily development workflow validation

---

In [0]:
%pip install pytest pytest-mock pytest-cov neo4j networkx pandas scikit-learn sentence-transformers streamlit --quiet

import time
import sys
from pathlib import Path

# ============================================================================
# PATH CONFIGURATION (Standard for all notebooks)
# ============================================================================

# Dynamic REPO_ROOT detection (works for any user)
try:
    # Try spark.conf first (works on Serverless)
    notebook_path = spark.conf.get("spark.databricks.notebook.path")
except:
    # Fallback to dbutils for classic compute
    try:
        notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    except:
        # Last resort - use current working directory
        notebook_path = str(Path.cwd())
        print("⚠️  Could not detect notebook path, using current directory")

# Convert notebook path to workspace path and derive repo root
# notebook_path is like: /Users/{username}/skillup/evaluation/notebooks/NotebookName
if notebook_path.startswith("/"):
    workspace_path = Path("/Workspace") / notebook_path.lstrip("/")
else:
    workspace_path = Path(notebook_path)

# Navigate up from notebooks -> evaluation -> skillup (repo root)
REPO_ROOT = workspace_path.parent.parent.parent if "notebooks" in str(workspace_path) else workspace_path

# Source data directory (version controlled)
DATA_DIR = REPO_ROOT / "data"

# Persistent artifact storage (Volumes - shared, not in git)
EVAL_ARTIFACTS = Path("/Volumes/workspace/default/iss-scratchpad/evaluation")
DATA_ARTIFACTS = Path("/Volumes/workspace/default/iss-scratchpad/data")

# Ensure artifact directories exist
EVAL_ARTIFACTS.mkdir(parents=True, exist_ok=True)
DATA_ARTIFACTS.mkdir(parents=True, exist_ok=True)

# Add skillup to Python path for imports
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("✅ Dependencies installed")
print(f"📁 REPO_ROOT: {REPO_ROOT}")
print(f"📁 DATA_DIR (source): {DATA_DIR}")
print(f"📦 EVAL_ARTIFACTS: {EVAL_ARTIFACTS}")
print(f"📦 DATA_ARTIFACTS: {DATA_ARTIFACTS}")

In [0]:
# Record start time
start_time = time.time()
test_results = []

print("⏱️  Starting smoke tests...")
print(f"🕐 Start time: {time.strftime('%H:%M:%S')}\n")

## 1. Critical Functionality Tests

Running pytest smoke tests across all modules...

In [0]:
# Run pytest smoke tests
import subprocess
import shutil
import os

print("🔥 Running pytest smoke tests...\n")

try:
    # Clean up __pycache__ directories to avoid filesystem issues
    pycache_dirs = list(REPO_ROOT.glob("**/__pycache__"))
    for pycache in pycache_dirs:
        try:
            shutil.rmtree(pycache)
        except:
            pass  # Ignore errors during cleanup
    
    # Set environment to prevent Python from creating __pycache__
    env = os.environ.copy()
    env['PYTHONDONTWRITEBYTECODE'] = '1'
    
    result = subprocess.run(
        ["python", "-m", "pytest", 
         str(REPO_ROOT / "tests"), 
         "-m", "smoke", 
         "-v", 
         "--tb=short",
         "-p", "no:cacheprovider"],  # Disable pytest cache
        capture_output=True,
        text=True,
        timeout=20,
        cwd=str(REPO_ROOT),
        env=env  # Use modified environment
    )
    
    print(result.stdout)
    if result.returncode == 0:
        test_results.append(("Pytest Smoke Tests", "✅ PASS"))
        print("\n✅ Pytest smoke tests PASSED")
    else:
        test_results.append(("Pytest Smoke Tests", "❌ FAIL"))
        print("\n❌ Pytest smoke tests FAILED")
        if result.stderr:
            print(f"Error: {result.stderr}")
except subprocess.TimeoutExpired:
    test_results.append(("Pytest Smoke Tests", "⚠️ TIMEOUT"))
    print("⚠️ Pytest smoke tests TIMEOUT (>20s)")
except Exception as e:
    test_results.append(("Pytest Smoke Tests", f"❌ ERROR: {str(e)}"))
    print(f"❌ Error running tests: {e}")

## 2. Module Import & Health Checks

Verifying each module can be imported and basic functionality works...

In [0]:
# Test 1: Import all modules
print("📦 Testing module imports...\n")

modules_to_test = [
    "knowledgegraph.knowledgegraph",
    "skillgap.skillgap",
    "recommender.recommender",
]

import_success = 0
import_failed = 0

for module_name in modules_to_test:
    try:
        __import__(module_name)
        print(f"✅ {module_name}")
        import_success += 1
    except Exception as e:
        print(f"❌ {module_name}: {str(e)[:80]}")
        import_failed += 1

if import_failed == 0:
    test_results.append(("Module Imports", "✅ PASS"))
    print(f"\n✅ All {import_success} modules imported successfully")
else:
    test_results.append(("Module Imports", f"⚠️ {import_failed} FAILED"))
    print(f"\n⚠️ {import_failed} modules failed to import")

In [0]:
# Test 2: Knowledge Graph basic functionality
print("\n🔗 Testing Knowledge Graph...\n")

try:
    from knowledgegraph.knowledgegraph import (
        get_skills_from_job,
        extract_all_role_skill_mappings,
        write_kg_output_to_delta
    )
    
    print("✅ Knowledge Graph module imported successfully")
    print("✅ Key functions available: get_skills_from_job, extract_all_role_skill_mappings")
    test_results.append(("KnowledgeGraph Functions", "✅ PASS"))
    
except ImportError as e:
    test_results.append(("KnowledgeGraph", "❌ IMPORT ERROR"))
    print(f"❌ Import error: {e}")

In [0]:
# Test 3: Skill Gap Analyzer basic functionality
print("\n📊 Testing Skill Gap Analyzer...\n")

try:
    from skillgap.skillgap import (
        find_skill_gaps,
        build_knowledge_graph,
        arbitrate_skill_gaps,
        load_user_profile
    )
    
    print("✅ Skill Gap module imported successfully")
    print("✅ Key functions available: find_skill_gaps, build_knowledge_graph")
    test_results.append(("SkillGap Import", "✅ PASS"))
        
except Exception as e:
    test_results.append(("SkillGap", f"❌ ERROR"))
    print(f"❌ SkillGap error: {str(e)[:100]}")

In [0]:
# Test 4: Recommender basic functionality
print("\n💡 Testing Recommender...\n")

try:
    from recommender.recommender import CourseRecommender
    
    # Test basic instantiation
    # Note: CourseRecommender requires config, so we test import only
    print("✅ CourseRecommender class imported successfully")
    test_results.append(("Recommender Import", "✅ PASS"))
    
    # Check if key classes exist
    from recommender.recommender import (
        ConstraintSolver,
        CaseLibrary,
        FuzzyScorer,
        ScoreFusion
    )
    print("✅ All recommender components available")
        
except Exception as e:
    test_results.append(("Recommender", f"❌ ERROR"))
    print(f"❌ Recommender error: {str(e)[:100]}")

In [0]:
# Test 5: Check gold standard data files exist
print("\n📁 Testing data files...\n")

data_files = [
    "data/gold_standard_cvs.json",
    "data/gold_standard_jds.json",
    "data/test_profiles.json",
    "data/skill_mappings_gold.json",
]

files_found = 0
files_missing = 0

for file_path in data_files:
    full_path = REPO_ROOT / file_path
    if full_path.exists():
        print(f"✅ {file_path}")
        files_found += 1
    else:
        print(f"❌ {file_path} (missing)")
        files_missing += 1

if files_missing == 0:
    test_results.append(("Data Files", "✅ PASS"))
    print(f"\n✅ All {files_found} data files found")
else:
    test_results.append(("Data Files", f"⚠️ {files_missing} MISSING"))
    print(f"\n⚠️ {files_missing} data files missing")

In [0]:
# Test 6: Check Databricks tables are accessible
print("\n🗄️  Testing Databricks tables...\n")

tables_to_check = [
    "workspace.default.my_skills_future_course_directory",
    "workspace.default.job_description",
    "workspace.default.resume_dataset_1200",
]

tables_accessible = 0
tables_failed = 0

try:
    for table_name in tables_to_check:
        try:
            # Try to get table count (minimal query)
            count = spark.sql(f"SELECT COUNT(*) as cnt FROM {table_name}").first().cnt
            print(f"✅ {table_name} ({count:,} rows)")
            tables_accessible += 1
        except Exception as te:
            print(f"❌ {table_name}: {str(te)[:80]}")
            tables_failed += 1
    
    if tables_failed == 0:
        test_results.append(("Databricks Tables", "✅ PASS"))
        print(f"\n✅ All {tables_accessible} tables accessible")
    else:
        test_results.append(("Databricks Tables", f"⚠️ {tables_failed} FAILED"))
        print(f"\n⚠️ {tables_failed} tables not accessible")
        
except Exception as e:
    test_results.append(("Databricks Tables", "❌ ERROR"))
    print(f"❌ Error checking tables: {str(e)[:100]}")

In [0]:
# Calculate elapsed time
end_time = time.time()
elapsed = end_time - start_time

# Display summary
print("\n" + "="*60)
print("🎯 SMOKE TEST RESULTS")
print("="*60)

passed = sum(1 for _, status in test_results if "✅" in status)
total = len(test_results)

for test_name, status in test_results:
    print(f"{status:<20} {test_name}")

print("="*60)
print(f"⏱️  Total time: {elapsed:.2f} seconds")
print(f"📊 Results: {passed}/{total} checks passed")

if elapsed < 30:
    print(f"✅ Target met: < 30 seconds")
else:
    print(f"⚠️  Target missed: {elapsed:.2f}s > 30s")

if passed == total:
    print("\n🎉 ALL SMOKE TESTS PASSED! System is healthy.")
elif passed >= total * 0.75:
    print(f"\n⚠️  MOSTLY PASSED ({passed}/{total}). Review warnings above.")
else:
    print(f"\n❌ MULTIPLE FAILURES ({total-passed}/{total}). System needs attention.")

print("="*60)

---

## ✅ Smoke Tests Complete!

**Next Steps:**
- If all tests passed: System is healthy, proceed with development
- If tests failed: Review errors above and fix issues before running full test suite
- For comprehensive testing: Run [Test_Runner](#notebook-984473343373211) notebook
- For coverage analysis: Run [Coverage_Analysis](#notebook-984473343373213) notebook

**Typical Issues:**
- ⚠️ Neo4j credentials: Expected in smoke tests (mocked functionality)
- ❌ Missing tables: Run data pipeline to populate Databricks tables
- ❌ Import errors: Check dependencies are installed
- ⚠️ Timeout: System may be slow, try running individual tests